# display() API Demo

BunBook provides a `display()` function for rendering rich outputs: HTML, Markdown, JSON, SVG, images, and more.


## HTML


In [ ]:
display.html(`
  <div style="padding: 24px; background: linear-gradient(135deg, #667eea, #764ba2, #f093fb); border-radius: 12px; color: white; font-family: sans-serif;">
    <h1 style="margin: 0 0 12px 0; font-size: 2em;">Hello from display.html()</h1>
    <p style="margin: 0 0 16px 0; font-size: 1.2em;">Supports any valid HTML — styles, tables, even inline SVG.</p>
    <div style="display: flex; gap: 12px;">
      <span style="background: rgba(255,255,255,0.25); padding: 8px 16px; border-radius: 20px; font-weight: bold;">TypeScript</span>
      <span style="background: rgba(255,255,255,0.25); padding: 8px 16px; border-radius: 20px; font-weight: bold;">Bun</span>
      <span style="background: rgba(255,255,255,0.25); padding: 8px 16px; border-radius: 20px; font-weight: bold;">BunBook</span>
    </div>
  </div>
`);


## Markdown


In [ ]:
display.markdown(`
### Features

- **Bold** and *italic* text
- [Links](https://bun.sh)
- Inline \`code\` and fenced blocks:

\`\`\`ts
const x = 42;
\`\`\`

| Column A | Column B |
|----------|----------|
| 1        | hello    |
| 2        | world    |
`);


## JSON


In [ ]:
display.json({
  name: "BunBook",
  version: Bun.version,
  features: ["TypeScript", "Plotly", "display() API"],
  config: { persistent: true, runtime: "bun" },
});


## SVG


In [ ]:
display.svg(`
<svg width="200" height="200" xmlns="http://www.w3.org/2000/svg">
  <rect width="200" height="200" rx="16" fill="#1a1a2e"/>
  <circle cx="60" cy="100" r="40" fill="#e94560"/>
  <circle cx="140" cy="100" r="40" fill="#0f3460" opacity="0.8"/>
  <circle cx="100" cy="100" r="20" fill="#16213e"/>
</svg>
`);


## Images (PNG from file)


In [ ]:
// Read a PNG file and display it
const png = await Bun.file("./gradient.png").arrayBuffer();
display.image(new Uint8Array(png));
console.log(`Displayed a ${png.byteLength}-byte PNG`);


## Image from the internet


In [ ]:
// Fetch an image from the web and display it
const response = await fetch("https://picsum.photos/200");
const buf = await response.arrayBuffer();
display.image(new Uint8Array(buf), "image/jpeg");
console.log(`Fetched ${buf.byteLength}-byte image from picsum.photos`);


## Mermaid diagram via CLI


In [ ]:
// Render a Mermaid diagram to PNG using mmdc (installed on the fly via bunx)
const sequence = Buffer.from(`
sequenceDiagram
  Client->>Server: POST /api/move
  Server->>Planner: computeTrajectory()
  Planner-->>Server: S-curve profile
  Server-->>Client: 200 OK
`);

const png =
  await Bun.$`bunx -p @mermaid-js/mermaid-cli mmdc -i - -o - -e png < ${sequence}`.arrayBuffer();
display.image(new Uint8Array(png));


In [ ]:
// Render a different Mermaid diagram to SVG
const flowchart = Buffer.from(`
graph TD
  A[Start] --> B{Decision}
  B -->|Yes| C[Do thing]
  B -->|No| D[Skip]
  C --> E[End]
  D --> E
`);

const svg =
  await Bun.$`bunx -p @mermaid-js/mermaid-cli mmdc -i - -o - -e svg < ${flowchart}`.text();
display.svg(svg);


## Matplotlib plot via Python


In [ ]:
// Generate a plot with matplotlib, output PNG to stdout
const script = `
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import sys

x = np.linspace(0, 4 * np.pi, 200)
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(x, np.sin(x), label='sin(x)')
ax.plot(x, np.cos(x), label='cos(x)')
ax.fill_between(x, np.sin(x), np.cos(x), alpha=0.1)
ax.set_title('Trigonometric Functions')
ax.legend()
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig(sys.stdout.buffer, format='png', dpi=150)
`;

const png = await Bun.$`python3 -c ${script}`.arrayBuffer();
display.image(new Uint8Array(png));


## Vega-Lite chart via npm


In [ ]:
// Render a Vega-Lite spec to SVG using vl2svg (installed on the fly via bunx)
const spec = JSON.stringify({
  $schema: "https://vega.github.io/schema/vega-lite/v5.json",
  width: 400,
  height: 200,
  data: {
    values: Array.from({ length: 50 }, (_, i) => ({
      x: i,
      sin: Math.sin(i / 5),
      cos: Math.cos(i / 5),
    })),
  },
  layer: [
    {
      mark: { type: "line", color: "#e45756" },
      encoding: {
        x: { field: "x", type: "quantitative" },
        y: { field: "sin", type: "quantitative", title: "value" },
      },
    },
    {
      mark: { type: "line", color: "#4c78a8" },
      encoding: {
        x: { field: "x", type: "quantitative" },
        y: { field: "cos", type: "quantitative" },
      },
    },
  ],
  title: "Sine & Cosine (Vega-Lite)",
});

const svg = await Bun.$`bunx -p vega-lite vl2svg < ${Buffer.from(spec)}`.text();
display.svg(svg);


## Generic display with MIME type


In [ ]:
// Pass raw data + explicit MIME type
display("<em>Italics via generic display()</em>", "text/html");


## Multi-MIME output


In [ ]:
// VS Code picks the richest MIME type it can render
display({
  "text/html": "<b>Rich HTML version</b>",
  "text/plain": "Plain text fallback",
});


## Plotly


In [ ]:
display.plotly(
  [
    {
      x: ["Mon", "Tue", "Wed", "Thu", "Fri"],
      y: [3, 7, 2, 8, 5],
      type: "bar",
      marker: { color: "#667eea" },
    },
  ],
  { title: "Weekly Activity", yaxis: { title: "Events" } }
);


## Multiple outputs in one cell


In [ ]:
console.log("1. console.log still works");
display.html("<b>2. HTML output</b>");
display.markdown("**3. Markdown output**");
console.log("4. More text after display calls");
